# 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import utils_z
import geopandas as gpd
import gdf_parser as gdfpar
import sql_utils
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
conn = utils_z.get_conn(os.getenv("DB_NAME"), os.getenv("DB_USER"), os.getenv("DB_PASSWORD"), os.getenv("DB_HOST"), os.getenv("DB_PORT"))

In [5]:
city_name = "madrid"
city_prefix  = "ES_MD"

# 1. 处理 shapefile 数据并导入数据库

## 1.1 查看数据

In [6]:
test_shp_path = rf"E:\2_data\building_3d_opensource\{city_name}\shp\CM1000_01_EDIFICIO_P.shp"
gdf = gpd.read_file(test_shp_path)

# 基本信息
print(f"CRS: {gdf.crs}")
print(f"总行数: {len(gdf)}")
print(f"字段名: {list(gdf.columns)}")
print(f"geometry类型: {gdf.geometry.type.unique()}")

# 前几行
print(gdf.head())

CRS: EPSG:25830
总行数: 148539
字段名: ['ID_3D', 'NIVEL', 'ORIGEN', 'FECHA_ALTA', 'SHAPE_STAr', 'SHAPE_STLe', 'geometry']
geometry类型: ['Polygon']
      ID_3D   NIVEL                      ORIGEN FECHA_ALTA  SHAPE_STAr  \
0  41008137  010103  RESTITUCION FOTOGRAMETRICA 2023-05-07  326.205415   
1  40058370  010103  RESTITUCION FOTOGRAMETRICA 2016-07-28  182.370000   
2  40025623  010103  RESTITUCION FOTOGRAMETRICA 2016-07-28   73.910000   
3  40106616  010103  RESTITUCION FOTOGRAMETRICA 2016-07-28   88.155186   
4  40002143  010103  RESTITUCION FOTOGRAMETRICA 2016-07-28   70.493000   

   SHAPE_STLe                                           geometry  
0   90.016074  POLYGON Z ((444646.663 4474570.576 689.678, 44...  
1   57.054943  POLYGON Z ((444067.5 4469145.9 624.09, 444065....  
2   35.435363  POLYGON Z ((436560.5 4472708.3 643.43, 436563....  
3   43.487475  POLYGON Z ((447987 4477532.9 657.62, 447982.6 ...  
4   51.751206  POLYGON Z ((440243.3 4466526.5 595.22, 440243....  


In [7]:
# 检查有没有高度相关字段
print(gdf.describe())

              ID_3D                  FECHA_ALTA     SHAPE_STAr     SHAPE_STLe
count  1.485390e+05                      148539  148539.000000  148539.000000
mean   4.010957e+07  2016-10-25 20:41:17.603000     340.957086      78.041497
min    0.000000e+00         2016-07-28 00:00:00       0.025100       1.029157
25%    4.003902e+07         2016-07-28 00:00:00      82.736300      39.674581
50%    4.007929e+07         2016-07-28 00:00:00     157.735000      57.166525
75%    4.011822e+07         2016-07-28 00:00:00     324.122600      90.679673
max    4.200156e+07         2024-08-14 00:00:00  224631.911500    9090.030291
std    1.287362e+06                         NaN    1255.837337      86.931188


In [18]:
gdf_4326 = gdf.to_crs(4326)
print(gdf_4326.geometry.iloc[0])
print(gdf_4326.geometry.bounds.describe())

POLYGON ((16.356862091395183 48.2085455476515, 16.356898703615737 48.20854083614809, 16.356887274552655 48.208501173744764, 16.356778930980497 48.208513967757966, 16.356798733845988 48.20844480674989, 16.356803919672625 48.20842671820741, 16.356810637038354 48.20840324178215, 16.356857259787944 48.20839683732379, 16.3568570576198 48.208396126817526, 16.35686659747618 48.20839486566694, 16.356855977805385 48.20835949336232, 16.35684681469934 48.208360691475086, 16.3568434183386 48.20834888268842, 16.356770678576453 48.208358305549886, 16.356778433567545 48.20833810283025, 16.356838215955587 48.20833074230225, 16.35681999422694 48.20826733639671, 16.35681810835116 48.20826292958784, 16.3568002127845 48.208265361695794, 16.356798326417444 48.20825987557387, 16.356741006675602 48.20826744244812, 16.356742826071688 48.20827362114374, 16.35672472846297 48.20827563055023, 16.356727558350975 48.20828462424814, 16.3567266836068 48.208284417557856, 16.356725795414686 48.20828422885882, 16.356724

In [8]:
# 看ID_3D是否有重复（同一建筑多行）
print(f"总行数：{len(gdf)}")
print(f"唯一ID_3D数：{gdf['ID_3D'].nunique()}")

# 看NIVEL的分布
print(gdf['NIVEL'].value_counts().head(20))

总行数：148539
唯一ID_3D数：148395
NIVEL
010103    148539
Name: count, dtype: int64


In [9]:
gdf['z_min'] = gdf.geometry.apply(lambda g: min(c[2] for c in g.exterior.coords))
gdf['z_max'] = gdf.geometry.apply(lambda g: max(c[2] for c in g.exterior.coords))
print(gdf[['z_min', 'z_max']].describe())

               z_min          z_max
count  148539.000000  148539.000000
mean      669.191461     669.203072
std        42.138471      42.067687
min         0.000000     556.690000
25%       632.620000     632.620000
50%       673.150000     673.150000
75%       703.300000     703.307450
max       980.780000     980.780000


In [20]:
gdf_override = gdf.copy()
gdf_override = gdf_override.set_crs(31256, allow_override=True)
gdf_4326 = gdf_override.to_crs(4326)
print(gdf_4326.geometry.bounds.describe())

              minx         miny         maxx         maxy
count  1260.000000  1260.000000  1260.000000  1260.000000
mean     16.355773    48.209493    16.355894    48.209576
std       0.002010     0.001262     0.001980     0.001257
min      16.352316    48.207400    16.352319    48.207401
25%      16.354091    48.208467    16.354259    48.208565
50%      16.355603    48.209422    16.355685    48.209503
75%      16.357994    48.210674    16.358034    48.210702
max      16.359019    48.211895    16.359043    48.211898


## 1.2 变量

In [15]:
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
#source_srid  = 3857

## 1.3 建表

In [ ]:
sql_utils.create_lod1_tables(
    conn=conn, 
    lod2_table_name=lod1_table_name,
    lod2_table_name_full=lod1_table_name_full, 
    lod2_surface_table_name=lod1_surface_table_name,
    lod2_surface_table_name_full=lod1_surface_table_name_full, 
    target_srid=target_srid
)

CH_ZU LOD1表创建完成


In [17]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.zurich_buildings_lod1表已清空
lod1.zurich_building_surfaces_lod1表已清空


In [11]:
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)

## 1.4 批量入库

In [18]:
shp_dir = rf"E:\2_data\building_3d_opensource\{city_name}\shp"

In [19]:
import traceback
import glob
conn.rollback()

In [ ]:
# 查找目录下所有shp文件
shp_files = glob.glob(rf"{shp_dir}\*.shp")
print(f"找到 {len(shp_files)} 个shp文件")

# 获取当前最大ID，设置计数器初始值
cur = conn.cursor()
cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1
cur.close()
print(f"building_counter起始值：{building_counter}")

total_count = 0
total_files = len(shp_files)

# 动态计算打印间隔（保证大约每10%的进度打印一次）
print_interval = max(1, total_files // 10)

# 依次处理每个shp文件
for idx, shp_path in enumerate(shp_files, 1):
    try:
        gdf = gpd.read_file(shp_path)
        #gdf.geometry = gdf.geometry.simplify(tolerance=0.000001, preserve_topology=True) # 简化几何，减少顶点数

        # 入库
        count, building_counter = gdfpar.insert_buildings_gdf_lod1(
            gdf, conn,
            lod1_table=lod1_table_name_full,
            city_prefix=city_prefix,
            building_counter=building_counter,
            col_height="h_rel_firs",
            col_ground_z="h_boden",
            col_citygml_id="egid"
        )
        
        total_count += count
        
        # 按间隔打印进度（以及最后一个文件）
        if idx % print_interval == 0 or idx == total_files:
            print(f"进度：{idx}/{total_files} 文件，已入库 {total_count} 栋建筑")
            
    except Exception as e:
        print(f"错误处理文件 {shp_path}：{e}")
        traceback.print_exc()

print(f"\n全部完成！总共处理 {total_files} 个文件，入库 {total_count} 栋建筑")

找到 1 个shp文件
building_counter起始值：1
进度：1/1 文件，已入库 65986 栋建筑

全部完成！总共处理 1 个文件，入库 65986 栋建筑


# 2. 叠合、建面

## 2.1 创建surface入库

In [ ]:
# # 先查当前surface表最大编号
# cur = conn.cursor()
# cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
# max_sid = cur.fetchone()[0]
# base = int(max_sid.split("_S_")[1]) if max_sid else 0
# cur.close()

# # GroundSurface
# utils_z.run_sql(f"""
#     INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
#     SELECT 
#         '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
#         building_id,
#         'GroundSurface',
#         ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z))
#     FROM {lod1_table_name_full};
# """, conn=conn)

# # 更新base
# cur = conn.cursor()
# cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
# max_sid = cur.fetchone()[0]
# base = int(max_sid.split("_S_")[1]) if max_sid else 0
# cur.close()

# # RoofSurface
# utils_z.run_sql(f"""
#     INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
#     SELECT 
#         '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
#         building_id,
#         'RoofSurface',
#         ST_Force3D(ST_Translate(geom_2d, 0, 0, ground_z + height))
#     FROM {lod1_table_name_full};
# """, conn=conn)

# # 更新base
# cur = conn.cursor()
# cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
# max_sid = cur.fetchone()[0]
# base = int(max_sid.split("_S_")[1]) if max_sid else 0
# cur.close()

# # WallSurface
# utils_z.run_sql(f"""
#     INSERT INTO {lod1_surface_table_name_full} (surface_id, building_id, surface_type, geom_3d)
#     SELECT
#         '{city_prefix}_S_' || LPAD((ROW_NUMBER() OVER () + {base})::TEXT, 8, '0'),
#         building_id,
#         'WallSurface',
#         ST_MakePolygon(ST_MakeLine(ARRAY[
#             ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z),
#             ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z),
#             ST_MakePoint(ST_X(p2), ST_Y(p2), ground_z + height),
#             ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z + height),
#             ST_MakePoint(ST_X(p1), ST_Y(p1), ground_z)
#         ]))
#     FROM (
#         SELECT
#             b.building_id,
#             b.ground_z,
#             b.height,
#             ST_PointN(ST_ExteriorRing(b.geom_2d), n) AS p1,
#             ST_PointN(ST_ExteriorRing(b.geom_2d), n + 1) AS p2
#         FROM {lod1_table_name_full} b,
#         GENERATE_SERIES(1, ST_NPoints(ST_ExteriorRing(b.geom_2d)) - 1) AS n
#     ) edges;
# """, conn=conn)

# # 统计
# cur = conn.cursor()
# cur.execute(f"""
#     SELECT surface_type, COUNT(*) 
#     FROM {lod1_surface_table_name_full} 
#     GROUP BY surface_type ORDER BY surface_type
# """)
# rows = cur.fetchall()
# for row in rows:
#     print(f"  {row[0]}: {row[1]}")

# # 计算总数
# total = sum(row[1] for row in rows)
# print(f"  Total: {total}")
# cur.close()

In [ ]:
gdfpar.generate_surfaces_from_buildings(
    conn,
    lod1_table=lod1_table_name_full,
    surface_table=lod1_surface_table_name_full,
    city_prefix=city_prefix
)

共65986栋建筑待生成surface
已插入surface：942139  GroundSurface: 65986
  RoofSurface: 65986
  WallSurface: 813226
  GroundSurface: 65986
  RoofSurface: 65986
  WallSurface: 813226
  Total: 945198


## 2.2 与block叠合

In [ ]:
sql_utils.map_buildings_to_blocks(
    block_table_name=block_table_name,
    building_table_name=lod1_table_name,
    building_table_name_full=lod1_table_name_full,
    conn=conn
)

# 3. 异常排查

## 3.1 中国数据顶点过多

In [31]:
gdf['vertex_count'] = gdf.geometry.apply(lambda g: len(g.exterior.coords))
print(gdf['vertex_count'].describe())
print(f"顶点数>20的建筑：{(gdf['vertex_count'] > 20).sum()}")
print(f"顶点数>50的建筑：{(gdf['vertex_count'] > 50).sum()}")
print(f"顶点数>100的建筑：{(gdf['vertex_count'] > 100).sum()}")

count    2.450940e+06
mean     2.347668e+01
std      1.218392e+01
min      5.000000e+00
25%      1.400000e+01
50%      2.100000e+01
75%      3.000000e+01
max      6.500000e+01
Name: vertex_count, dtype: float64
顶点数>20的建筑：1229883
顶点数>50的建筑：103265
顶点数>100的建筑：0
